# Memory X-Ray: re-run the full experiment on a GPU (optional)

**You do not need this notebook to explore or check the published results.** Everything in the report can be
reproduced on a CPU from the saved run: see `notebooks/explore_results.ipynb` and `python scripts/verify_release.py`.

This notebook re-runs the GPU experiment from scratch with the same frozen scenarios, pinned model commit and settings,
then compares the new run with the published one.

| | |
|---|---|
| Hardware | Google Colab with a **T4 GPU** (Runtime → Change runtime type → T4 GPU) |
| Time | about 40 minutes for the run (the published run took 2,225.6 s) plus the model download |
| Model | `microsoft/Phi-3-mini-4k-instruct` @ `f39ac1d28e925b323eae81227eaba4464caced4e`, downloaded from Hugging Face under the license on its model card (weights are not part of this repository) |
| Output | a new folder `outputs/runs/full_<UTC time>/`; the published run in `runs/` is never touched |

**What to expect.** The 48 scored prompts per scenario are deterministic given the hardware and library versions, so a
re-run on a T4 with the recorded versions should give the same decisions and check outcomes with at most tiny numerical
differences. Other GPUs or library versions can shift probabilities slightly. The 60 sampled citation replies
(temperature 1.0) are not expected to repeat exactly.

The runner refuses to start unless gates G1–G4 hold: frozen files unchanged (G1), the two rubric label files valid and
in agreement (G2; these are AI-assisted labels, not human annotation), an explicit approval name (G3), and the pinned
model commit verified (G4).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

Either clone the repository (default) or set `USE_UPLOAD = True` and upload the release ZIP
(`memory-xray-github-final.zip`). An existing copy is moved aside, never deleted.

In [ ]:
import os, subprocess, time, zipfile
REPO_URL = "https://github.com/FarnooshMemari/memory-xray"   # update if the repository name differs
USE_UPLOAD = False
TARGET = "memory-xray"

os.chdir("/content")
if os.path.isdir(TARGET):
    os.rename(TARGET, f"{TARGET}_old_{int(time.time())}")
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    zipfile.ZipFile(next(iter(uploaded))).extractall("_upload")
    top = next(os.path.join(dp) for dp, dn, fn in os.walk("_upload") if "run_experiment.py" in fn)
    os.rename(top, TARGET)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, TARGET], check=True)
os.chdir(TARGET)
import sys; sys.path.insert(0, os.getcwd())
import mxr
assert mxr.__version__ == "0.2.4", f"unexpected package version {mxr.__version__}"
print("code ready:", os.getcwd(), "| mxr", mxr.__version__)

## 2. Install and check

`transformers` is pinned to the version recorded in the published run (5.18.0). Colab's own PyTorch build is kept; its
version is printed and recorded in the new run's manifest. Then the offline tests run (tiny random models, no download),
followed by gates G1 and G2 and the release verification (all CPU).

In [ ]:
!pip install -q "transformers==5.18.0" "accelerate>=0.31" "pytest>=7" "matplotlib>=3.7"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())
print("published run used: torch 2.11.0+cu130, transformers 5.18.0, Python 3.13.15")

In [ ]:
!python -m pytest -q
!python validate_scenarios.py --verify-freeze
!python check_labels.py
!python scripts/verify_release.py

## 3. Approval (G3)

Type the name of the person approving this run; it is recorded in the run's manifest. Leave it empty to stop here.

In [ ]:
APPROVED_BY = ""   # your name
assert APPROVED_BY.strip(), "Not approved: set APPROVED_BY to run the full experiment."

## 4. Run the full experiment (GPU, about 40 minutes)

In [ ]:
!python run_experiment.py --model microsoft/Phi-3-mini-4k-instruct --revision f39ac1d28e925b323eae81227eaba4464caced4e --storage-dtype float16 --compute-dtype float32 --confirm-full-run --approved-by "$APPROVED_BY" --out outputs/runs

## 5. Compare with the published run

`scripts/compare_runs.py` reports, per scenario, whether all 48 prompts are present, whether the decision d\*, the
per-ordering answers, the flips and every check outcome match the published run, and the largest differences in
probabilities and Shapley values (accepted up to 0.01 by default). `reanalyze.py` recomputes the new run's checks from
its saved files (output in `outputs/reanalysis/`).

In [ ]:
import glob
# newest run folder (run folders contain manifest.json; comparison files and reanalysis outputs do not)
run_dir = sorted(d for d in glob.glob("outputs/runs/full_*") if os.path.isfile(os.path.join(d, "manifest.json")))[-1]
print(run_dir)
!python scripts/compare_runs.py {run_dir} --json {run_dir}_comparison.json
!python reanalyze.py {run_dir}

## 6. View the summary and download the new run

In [ ]:
import shutil
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(run_dir, "SUMMARY.md")).read()))
archive = shutil.make_archive(run_dir, "zip", run_dir)
from google.colab import files
files.download(archive)
files.download(f"{run_dir}_comparison.json")